In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings



warnings.filterwarnings('ignore')

In [4]:
books = pd.read_csv('../data/BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv('../data/BX-Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
ratings = pd.read_csv('../data/BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

In [5]:
print("Books shape:", books.shape)
print("Users shape:", users.shape)
print("Ratings shape:", ratings.shape)

Books shape: (271360, 8)
Users shape: (278858, 3)
Ratings shape: (1149780, 3)


In [6]:
print(books.columns)
books.head(3)

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='str')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...


In [7]:
print(users.columns)
users.head(3)

Index(['User-ID', 'Location', 'Age'], dtype='str')


,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


In [8]:
print(ratings.columns)
ratings.head(3)

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='str')


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


In [ ]:
# Rename Books columns
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]
books.rename(columns={
    'Book-Title': 'title',
    'Book-Author': 'author',
    'Year-Of-Publication': 'year',
    'Publisher': 'publisher',
    'Image-URL-L': 'image_url'
}, inplace=True)

In [10]:
# Rename Users columns
users.rename(columns={
    'User-ID': 'user_id',
    'Location': 'location',
    'Age': 'age'
}, inplace=True)

In [11]:
# Rename Ratings columns
ratings.rename(columns={
    'User-ID': 'user_id',
    'Book-Rating': 'rating'
}, inplace=True)

In [12]:
print(books.columns.tolist())
print(users.columns.tolist())
print(ratings.columns.tolist())

['ISBN', 'title', 'author', 'year', 'publisher', 'image_url']
['user_id', 'location', 'age']
['user_id', 'ISBN', 'rating']


Next Step: Basic understanding of the data
Run these cells one by one:
1. Check rating distribution

In [13]:
ratings['rating'].value_counts().sort_index()

rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64

In [14]:
print("Unique users who rated:", ratings['user_id'].nunique())
print("Unique books that got rated:", ratings['ISBN'].nunique())

Unique users who rated: 105283
Unique books that got rated: 340556


In [15]:
print("Books missing values:\n", books.isnull().sum())
print("\nUsers missing values:\n", users.isnull().sum())
print("\nRatings missing values:\n", ratings.isnull().sum())

Books missing values:
 ISBN         0
title        0
author       2
year         0
publisher    2
image_url    3
dtype: int64

Users missing values:
 user_id          0
location         0
age         110762
dtype: int64

Ratings missing values:
 user_id    0
ISBN       0
rating     0
dtype: int64


Next Important Step: Filter data for better recommendations
In collaborative filtering, we should only keep:

Users who have rated a decent number of books (active users)
Books that have been rated by a decent number of users (popular books)

This removes noise and makes the model better + faster.
Run this cell:

In [16]:
# Count how many ratings each user has given
user_rating_count = ratings['user_id'].value_counts()

# Keep only users who have rated at least 200 books
active_users = user_rating_count[user_rating_count >= 200].index
ratings = ratings[ratings['user_id'].isin(active_users)]

print("Ratings after filtering active users:", ratings.shape)

Ratings after filtering active users: (527556, 3)


In [17]:
# Count how many ratings each book has received
book_rating_count = ratings['ISBN'].value_counts()

# Keep only books that have been rated at least 50 times
popular_books = book_rating_count[book_rating_count >= 50].index
ratings = ratings[ratings['ISBN'].isin(popular_books)]

print("Ratings after filtering popular books:", ratings.shape)
print("Unique users left:", ratings['user_id'].nunique())
print("Unique books left:", ratings['ISBN'].nunique())

Ratings after filtering popular books: (42093, 3)
Unique users left: 889
Unique books left: 526


# Next Step: Merge ratings with books
 # We need book title and image URL later, so we merge.


In [22]:
# Merge ratings with books on ISBN
final_rating = ratings.merge(books, on='ISBN')

print("Shape after merge:", final_rating.shape)
final_rating.head(3)

Shape after merge: (41914, 8)


,user_id,ISBN,rating,title,author,year,publisher,image_url
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,http://images.amazon.com/images/P/0060930535.0...
2,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,http://images.amazon.com/images/P/0060934417.0...


In [23]:
# Create pivot table
book_pivot = final_rating.pivot_table(
    columns='user_id',
    index='title',
    values='rating'
)

print("Pivot shape:", book_pivot.shape)
book_pivot.head()

Pivot shape: (515, 889)


user_id,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1st to Die: A Novel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2nd Chance,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4 Blondes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A Bend in the Road,NaN,NaN,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
#Cell 1 – Fill NaNs with 0
book_pivot.fillna(0, inplace=True)
book_pivot.head(3)

user_id,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
# Cell 2 – Train Nearest Neighbors model

from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(algorithm='brute', metric='cosine')
model.fit(book_pivot)

,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


Next Step: Create the recommendation function
Paste and run this cell:

In [26]:
def recommend_books(book_name):
    # Find the index of the book
    book_id = np.where(book_pivot.index == book_name)[0][0]
    
    # Get distances and indices of nearest neighbors
    distances, suggestions = model.kneighbors(
        book_pivot.iloc[book_id, :].values.reshape(1, -1),
        n_neighbors=6
    )
    
    print(f"\nBooks similar to '{book_name}':\n")
    
    for i in range(len(suggestions)):
        books = book_pivot.index[suggestions[i]]
        for j in books:
            if j != book_name:          # skip the same book
                print("→", j)

In [27]:
recommend_books("1984")


Books similar to '1984':

→ The Handmaid's Tale
→ The Catcher in the Rye
→ Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death
→ The Vampire Lestat (Vampire Chronicles, Book II)
→ The Tale of the Body Thief (Vampire Chronicles (Paperback))


In [28]:
recommend_books("The Da Vinci Code")


Books similar to 'The Da Vinci Code':

→ Angels &amp; Demons
→ Touching Evil
→ The Blue Nowhere : A Novel
→ The Testament
→ The Lovely Bones: A Novel


In [ ]:
'''# Next Step: Save the necessary files
# We need to save 4 things so the Streamlit app can use them later:

# The trained model
# The list of book names
# The final_rating dataframe (for image URLs)
# The book_pivot matrix'''

'# Next Step: Save the necessary files\n# We need to save 4 things so the Streamlit app can use them later:\n\n# The trained model\n# The list of book names\n# The final_rating dataframe (for image URLs)\n# The book_pivot matrix'

In [30]:
import pickle
import os

# Create artifacts folder if it doesn't exist (safety)
os.makedirs('../artifacts', exist_ok=True)

# Save everything
pickle.dump(model, open('../artifacts/model.pkl', 'wb'))
pickle.dump(book_pivot.index, open('../artifacts/book_names.pkl', 'wb'))
pickle.dump(final_rating, open('../artifacts/final_rating.pkl', 'wb'))
pickle.dump(book_pivot, open('../artifacts/book_pivot.pkl', 'wb'))

print("All artifacts saved successfully!")

All artifacts saved successfully!
